# 04 - Report assets

Generate the figures and tables used in the report. No model training.

| Section | Output |
|---|---|
| S6a | CPU vs GPU throughput table |
| S6b | fp32 vs AMP timing and peak memory |
| S6c | Architecture ablation and capacity-scaling plots |
| S6d | Confusion matrices, per-class accuracy, error statistics |
| S6e | Grid of highest-confidence errors |

Each section is self-contained: S6e reads the `error_samples.npz` written by S6d.
Figures go to `figs/`; S6d also updates `task_a/runs_final/`.

**How to run:** run the Setup cell once, then any section. S6a/S6b require CUDA;
S6d benefits from a GPU; S6c and S6e only read saved files.


## Setup - locate `task_a/src`

In [2]:
# Locate task_a/src so `import src...` works whether the kernel starts in
# task_a/notebooks or at the repo root.
import pathlib, sys

CWD = pathlib.Path.cwd().resolve()
ROOT = next((p for p in (CWD, *CWD.parents) if (p / "task_a" / "src").is_dir()), None)
assert ROOT is not None, "cannot locate task_a/src from %s" % CWD
sys.path.insert(0, str(ROOT / "task_a"))

## S6a - CPU vs GPU throughput

Time one forward + backward + step of `cnn_down` on CPU and GPU
(batch 128, 3 measured steps after 1 warmup).


In [3]:
# S6a: CPU vs GPU training throughput (~10 s).
import time
import torch
from src.models import cnn_down

BS, STEPS, WARM = 128, 3, 1


def bench(factory, device):
    torch.manual_seed(0)
    model = factory().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = torch.nn.CrossEntropyLoss()
    x = torch.randn(BS, 3, 32, 32, device=device)
    y = torch.randint(0, 100, (BS,), device=device)

    def step():
        opt.zero_grad()
        crit(model(x), y).backward()
        opt.step()

    for _ in range(WARM):
        step()
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(STEPS):
        step()
    if device.type == "cuda":
        torch.cuda.synchronize()
    dt = (time.time() - t0) / STEPS
    return dt, BS / dt


devices = [("CPU", torch.device("cpu"))]
if torch.cuda.is_available():
    devices.append(("GPU", torch.device("cuda")))

res = {}
print("%-6s %14s %14s" % ("device", "ms/step", "img/s"))
for name, dev in devices:
    dt, ips = bench(cnn_down, dev)
    res[name] = ips
    print("%-6s %14.1f %14.0f" % (name, dt * 1000, ips))
if len(res) == 2:
    print()
    print("GPU speed-up vs CPU: %.1fx" % (res["GPU"] / res["CPU"]))


device        ms/step          img/s
CPU             191.0            670
GPU              28.1           4559

GPU speed-up vs CPU: 6.8x


## S6b - fp32 vs automatic mixed precision (AMP)

Engineering comparison only: AMP changes numerics, so it is not part of any controlled
experiment. Time fp32 and autocast+fp16 for the 20.7M champion and record peak memory.


In [4]:
# S6b: fp32 vs AMP step time and peak memory (~30 s). Engineering only; not part of the controlled experiments.
import time
import torch
from src.models import cnn_deep_residual_wide

BS, STEPS, WARM = 128, 5, 2


def bench(use_amp):
    torch.manual_seed(0)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model = cnn_deep_residual_wide().to("cuda")
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = torch.nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda") if use_amp else None
    x = torch.randn(BS, 3, 32, 32, device="cuda")
    y = torch.randint(0, 100, (BS,), device="cuda")

    def step():
        opt.zero_grad()
        if use_amp:
            with torch.autocast("cuda", dtype=torch.float16):
                loss = crit(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            crit(model(x), y).backward()
            opt.step()

    for _ in range(WARM):
        step()
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(STEPS):
        step()
    torch.cuda.synchronize()
    return (time.time() - t0) / STEPS, torch.cuda.max_memory_allocated() / 2 ** 20


fp32_ms, fp32_mem = bench(False)
amp_ms, amp_mem = bench(True)
print("fp32: %6.0f ms/step | peak %.0f MB" % (fp32_ms * 1000, fp32_mem))
print("AMP : %6.0f ms/step | peak %.0f MB" % (amp_ms * 1000, amp_mem))
print()
print("speed-up: %.2fx | memory saved: %.0f MB" % (fp32_ms / amp_ms, fp32_mem - amp_mem))
print("Note: AMP is an engineering comparison only; all experiment numbers use fp32.")


fp32:    170 ms/step | peak 1379 MB
AMP :     91 ms/step | peak 861 MB

speed-up: 1.88x | memory saved: 518 MB
Note: AMP is an engineering comparison only; all experiment numbers use fp32.


## S6c - Ablation and capacity plots

Read `metrics.csv` from `task_a/runs/` (60 epochs, 3 seeds); no re-evaluation.

- Left: architecture delta vs `exp_221_base_down60` (69.40), sorted by gain.
- Right: augmentation effect at fixed architecture and budget.
- Second figure: capacity ladders (parameters vs validation accuracy) for width and depth.


In [5]:
# S6c: ablation bar chart + capacity ladder (no GPU, ~5 s). Writes to figs/.
import csv
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from src.config import PROJECT_ROOT, RUNS_DIR

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "font.size": 9,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#0b0b0b",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
})
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
RED, MUTED, INK = "#e34948", "#898781", "#0b0b0b"
FIG = PROJECT_ROOT / "figs"
FIG.mkdir(exist_ok=True)


def final_accs(exp):
    out = []
    for s in (0, 42, 123):
        p = RUNS_DIR / ("%s_s%d" % (exp, s)) / "metrics.csv"
        rows = list(csv.DictReader(open(p, newline="", encoding="utf-8")))
        out.append(float(rows[-1]["test_acc"]) * 100)
    return np.array(out)


base = final_accs("exp_221_base_down60")
BASE = base.mean()
BASE_SD = base.std()

# ---- Left: architecture delta vs the 60-epoch base, sorted by gain ----
items = [("wide x2", "exp_212_wide_aug"), ("wide x3", "exp_219_wide3_aug"),
         ("dr (d2+res)", "exp_214_deepres_aug"), ("dr d3", "exp_217_dr_d3_aug"),
         ("dr d4", "exp_218_dr_d4_aug"), ("dr_wide", "exp_216_dr_wide_aug"),
         ("wide+Cutout", "exp_220_wide_cutout")]
rows = []
for label, exp in items:
    a = final_accs(exp)
    delta = a.mean() - BASE
    err = np.sqrt(a.std() ** 2 + BASE_SD ** 2)
    rows.append((label, a.mean(), delta, err))
rows.sort(key=lambda r: r[2])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.6, 1]})

labels = [r[0] for r in rows]
deltas = [r[2] for r in rows]
errs = [r[3] for r in rows]
colors = [BLUE if d >= 0 else RED for d in deltas]
y = np.arange(len(rows))
ax1.barh(y, deltas, xerr=errs, color=colors, height=0.62,
         error_kw={"ecolor": MUTED, "elinewidth": 1, "capsize": 2})
ax1.set_yticks(y, labels)
ax1.axvline(0, color=MUTED, linewidth=0.8)
for yi, d in zip(y, deltas):
    ax1.text(d + (0.12 if d >= 0 else -0.12), yi, "%+.2f" % d,
             va="center", ha="left" if d >= 0 else "right", fontsize=8, color=INK)
ax1.set_xlabel("delta val top-1 vs base (pt)")
ax1.set_title("Architecture factors @ 60 ep (base = 69.40)", loc="left")
ax1.set_xlim(min(0, min(deltas) - 0.5), max(deltas) + 0.8)

# ---- Right: augmentation effect (same architecture, same budget) ----
pairs = [("wide x2", "exp_211_wide_noaug", "exp_212_wide_aug"),
         ("dr (d2+res)", "exp_213_deepres_noaug", "exp_214_deepres_aug")]
x = np.arange(len(pairs))
w = 0.34
noaug = [final_accs(p[1]).mean() for p in pairs]
aug = [final_accs(p[2]).mean() for p in pairs]
ax2.bar(x - w / 2, noaug, w, label="no aug", color=ORANGE)
ax2.bar(x + w / 2, aug, w, label="crop+flip", color=BLUE)
for xi, a, b in zip(x, noaug, aug):
    ax2.text(xi - w / 2, a + 0.4, "%.1f" % a, ha="center", fontsize=8, color=INK)
    ax2.text(xi + w / 2, b + 0.4, "%.1f" % b, ha="center", fontsize=8, color=INK)
    ax2.text(xi, max(a, b) + 3.0, "+%.1f" % (b - a), ha="center", fontsize=9, color=BLUE)
ax2.set_xticks(x, [p[0] for p in pairs])
ax2.set_ylim(0, 85)
ax2.set_ylabel("val top-1 (%)")
ax2.set_title("Augmentation effect @ 60 ep", loc="left")
ax2.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2)
ax2.grid(axis="x", visible=False)

fig.tight_layout()
fig.savefig(FIG / "fig_ablation_60ep.png", bbox_inches="tight")
plt.close(fig)

# ---- Capacity ladder: parameter count vs val ----
width_ladder = [("wx1", 2.35, "exp_221_base_down60"), ("wx2", 9.36, "exp_212_wide_aug"),
                ("wx3", 21.00, "exp_219_wide3_aug")]
depth_ladder = [("d1", 2.35, "exp_221_base_down60"), ("d2", 5.19, "exp_214_deepres_aug"),
                ("d3", 7.93, "exp_217_dr_d3_aug"), ("d4", 10.66, "exp_218_dr_d4_aug")]
combined = ("dr_wide", 20.70, "exp_216_dr_wide_aug")

fig, ax = plt.subplots(figsize=(7.2, 4.2))
LABELS = {
    (2.35, 69.40): ("base 69.40", (0, -15), "center"),
    (9.36, 72.48): ("wx2 72.48", (0, 10), "center"),
    (21.00, 73.40): ("wx3 73.40", (-10, -13), "right"),
    (5.19, 71.46): ("d2 71.46", (0, 10), "center"),
    (7.93, 71.98): ("d3 71.98", (5, -14), "left"),
    (10.66, 71.99): ("d4 71.99", (9, 7), "left"),
}
drawn = set()
for ladder, color, ls, marker, name in ((width_ladder, BLUE, "-", "o", "width ladder (depth 1, no res)"),
                                        (depth_ladder, ORANGE, "--", "s", "depth ladder (width x1, +res)")):
    xs = [p[1] for p in ladder]
    ys = [final_accs(p[2]).mean() for p in ladder]
    ax.plot(xs, ys, color=color, linestyle=ls, marker=marker, markersize=6,
            linewidth=1.6, label=name)
    for (lab, x, exp), yv in zip(ladder, ys):
        key = (x, round(yv, 2))
        if key in drawn:
            continue
        drawn.add(key)
        text, off, ha = LABELS.get(key, ("%s %.2f" % (lab, yv), (0, -13), "center"))
        ax.annotate(text, (x, yv), textcoords="offset points", xytext=off, ha=ha,
                    fontsize=7.5, color=color)
xc, yc = combined[1], final_accs(combined[2]).mean()
ax.scatter([xc], [yc], color=AQUA, marker="*", s=180, zorder=5,
           label="combined: wide x2 + d2 + res")
ax.annotate("dr_wide %.2f" % yc, (xc, yc), textcoords="offset points",
            xytext=(0, 9), ha="center", fontsize=8, color=AQUA)
ax.set_xlim(1.5, 22.5)
ax.margins(y=0.12)
ax.set_xlabel("parameters (M)")
ax.set_ylabel("val top-1 (%)")
ax.set_title("Capacity scaling @ 60 ep (3 seeds)", loc="left")
ax.legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(FIG / "fig_capacity_scaling.png", bbox_inches="tight")
plt.close(fig)

print("wrote", FIG / "fig_ablation_60ep.png")
print("wrote", FIG / "fig_capacity_scaling.png")


wrote C:\Users\sigridwicks\Desktop\DL HW1\figs\fig_ablation_60ep.png
wrote C:\Users\sigridwicks\Desktop\DL HW1\figs\fig_capacity_scaling.png


## S6d - Error analysis of the final 3-seed ensemble

Average the probabilities of the three 50k models in `runs_final/` and evaluate on test10k.
Produces:

- a 100x100 confusion matrix (row-normalised) and a 20-superclass aggregate.
  Fine names are alphabetical, so map each fine class to its superclass via `coarse_labels`;
  `i // 5` is wrong.
- per-class accuracy (sorted);
- the share of errors within a superclass and the most confused class pairs;
- error samples saved to `error_samples.npz` for S6e.

The models are frozen; this section only analyses them.


In [6]:
# S6d: confusion matrix / per-class / error statistics for the final models (~1-2 min). Writes figs/ and runs_final/.
import pickle
import pathlib
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from src.config import DATA_DIR, PROJECT_ROOT, RUNS_DIR
from src.data import build_loaders
from src.models import cnn_deep_residual_wide

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "font.size": 9,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#0b0b0b",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.spines.top": False, "axes.spines.right": False,
})
FIG = PROJECT_ROOT / "figs"
FIG.mkdir(exist_ok=True)
RUNS_FINAL = RUNS_DIR.parent / "runs_final"
SEQ = LinearSegmentedColormap.from_list(
    "seq_blue", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])

# Class names and the fine-to-superclass map. Fine names are alphabetical, so i // 5 is not the superclass;
# derive the true map from (fine_labels, coarse_labels) in the dataset.
with open(DATA_DIR / "cifar-100-python" / "meta", "rb") as f:
    meta = pickle.load(f, encoding="latin1")
fine_names = [str(x) for x in meta["fine_label_names"]]
coarse_names = [str(x) for x in meta["coarse_label_names"]]

with open(DATA_DIR / "cifar-100-python" / "test", "rb") as f:
    _raw = pickle.load(f, encoding="latin1")
_fine = np.array(_raw["fine_labels"])
_coarse = np.array(_raw["coarse_labels"])
fine2coarse = np.zeros(100, dtype=int)
for _fi in range(100):
    _vals = np.unique(_coarse[_fine == _fi])
    assert len(_vals) == 1, "fine class %d maps to several superclasses" % _fi
    fine2coarse[_fi] = _vals[0]
assert len(set(fine2coarse.tolist())) == 20, "expected 20 superclasses"

_, test_loader = build_loaders(batch_size=128, num_workers=2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models = []
for s in (0, 42, 123):
    m = cnn_deep_residual_wide().to(device)
    m.load_state_dict(torch.load(RUNS_FINAL / ("exp_230_final_dr_wide_s%d" % s) / "model.pt",
                                 map_location=device, weights_only=True))
    m.eval()
    models.append(m)

cm = np.zeros((100, 100), dtype=np.int64)
n_top1 = n_top5 = n_total = 0
err = []            # (confidence, image, true, pred)
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs_gpu = imgs.to(device)
        prob = 0
        for m in models:
            prob = prob + m(imgs_gpu).softmax(dim=1)
        prob = prob / len(models)

        conf, pred = prob.max(dim=1)
        top5 = prob.topk(5, dim=1).indices
        n_top1 += int((pred.cpu() == labels).sum())
        n_top5 += int((top5.cpu() == labels[:, None]).any(dim=1).sum())
        n_total += int(labels.size(0))

        for t, p in zip(labels.tolist(), pred.cpu().tolist()):
            cm[t, p] += 1
        wrong = pred.cpu() != labels
        for c, im, t, p in zip(conf.cpu()[wrong].tolist(), imgs[wrong], labels[wrong].tolist(),
                               pred.cpu()[wrong].tolist()):
            err.append((c, im, t, p))

print("ensemble test top-1 %.4f | top-5 %.4f | n=%d" % (n_top1 / n_total, n_top5 / n_total, n_total))
print("misclassified:", len(err))

# ---- Superclass aggregation ----
cm_coarse = np.zeros((20, 20), dtype=np.int64)
for i in range(100):
    for j in range(100):
        cm_coarse[fine2coarse[i], fine2coarse[j]] += cm[i, j]
row = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, row, out=np.zeros_like(cm, dtype=float), where=row > 0)
crow = cm_coarse.sum(axis=1, keepdims=True)
cm_coarse_norm = np.divide(cm_coarse, crow, out=np.zeros_like(cm_coarse, dtype=float), where=crow > 0)

within = sum(cm[i, j] for i in range(100) for j in range(100) if i != j and fine2coarse[i] == fine2coarse[j])
all_err = int(cm.sum() - np.trace(cm))
print("errors within the same superclass: %d / %d = %.1f%%" % (within, all_err, 100 * within / all_err))

# ---- Per-class accuracy ----
per_class = np.diag(cm) / np.maximum(cm.sum(axis=1), 1)
order = np.argsort(per_class)
print()
print("worst 10 classes:")
for i in order[:10]:
    print("  %-16s %.3f  (n=%d)" % (fine_names[i], per_class[i], cm[i].sum()))
print("best 5 classes:")
for i in order[::-1][:5]:
    print("  %-16s %.3f" % (fine_names[i], per_class[i]))

# ---- Most confused class pairs ----
pairs = [(cm[i, j], i, j) for i in range(100) for j in range(100) if i != j]
pairs.sort(reverse=True)
print()
print("top confused pairs (true -> predicted):")
for cnt, i, j in pairs[:10]:
    print("  %-16s -> %-16s %d" % (fine_names[i], fine_names[j], cnt))

# ---- Figure 1: fine 100x100 ----
fig, ax = plt.subplots(figsize=(6.6, 5.6))
im = ax.imshow(cm_norm, cmap=SEQ, vmin=0, vmax=0.3, interpolation="nearest")
ax.set_xlabel("predicted class"); ax.set_ylabel("true class")
ax.set_title("Confusion matrix (row-normalised, saturated at 0.3)", loc="left", fontsize=9)
ticks = list(range(0, 100, 10))
ax.set_xticks(ticks); ax.set_yticks(ticks)
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIG / "fig_confusion_fine.png", bbox_inches="tight")
plt.close(fig)

# ---- Figure 2: 20 superclasses ----
fig, ax = plt.subplots(figsize=(7.6, 6.4))
im = ax.imshow(cm_coarse_norm, cmap=SEQ, vmin=0, vmax=0.6, interpolation="nearest")
ax.set_xticks(range(20), coarse_names, rotation=90, fontsize=7)
ax.set_yticks(range(20), coarse_names, fontsize=7)
ax.set_xlabel("predicted superclass"); ax.set_ylabel("true superclass")
title = ("Confusion matrix, 20 superclasses (%d%% of errors stay inside one superclass)"
         % round(100 * within / all_err))
ax.set_title(title, loc="left", fontsize=9)
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIG / "fig_confusion_coarse.png", bbox_inches="tight")
plt.close(fig)

# ---- Figure 3: per-class accuracy ----
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.bar(np.arange(100), per_class[order] * 100, color="#2a78d6", width=0.85)
ax.set_xlim(-0.6, 99.6)
ax.set_xticks(range(0, 100, 5), [fine_names[i] for i in order[::5]], rotation=90, fontsize=6)
ax.set_ylabel("accuracy (%)")
ax.set_title("Per-class accuracy on test10k (sorted, mean %.1f%%)" % (100 * per_class.mean()), loc="left")
ax.grid(axis="x", visible=False)
i = order[0]
ax.annotate("worst: %s %.0f%%" % (fine_names[i], 100 * per_class[i]),
            (0, 100 * per_class[i]), textcoords="offset points", xytext=(12, 4),
            fontsize=7, color="#0b0b0b")
fig.tight_layout()
fig.savefig(FIG / "fig_per_class.png", bbox_inches="tight")
plt.close(fig)

# ---- Save: per-class CSV + error-sample npz ----
with open(RUNS_FINAL / "per_class_acc.csv", "w", newline="", encoding="utf-8") as f:
    import csv
    w = csv.writer(f); w.writerow(["class", "superclass", "n", "accuracy"])
    for i in range(100):
        w.writerow([fine_names[i], coarse_names[fine2coarse[i]], int(cm[i].sum()), round(float(per_class[i]), 4)])

err.sort(key=lambda t: -t[0])
pick = err[:12]
np.savez(RUNS_FINAL / "error_samples.npz",
         imgs=np.stack([t[1].numpy() for t in pick]),
         true=np.array([t[2] for t in pick]), pred=np.array([t[3] for t in pick]),
         conf=np.array([t[0] for t in pick]), fine_names=np.array(fine_names))
np.savez(RUNS_FINAL / "confusion.npz", fine=cm, coarse=cm_coarse,
         per_class=per_class, fine_names=np.array(fine_names),
         coarse_names=np.array(coarse_names))

print()
print("wrote figs/fig_confusion_fine.png, figs/fig_confusion_coarse.png, figs/fig_per_class.png")
print("wrote", RUNS_FINAL / "per_class_acc.csv", "|", RUNS_FINAL / "confusion.npz")


ensemble test top-1 0.7757 | top-5 0.9412 | n=10000
misclassified: 2243
errors within the same superclass: 812 / 2243 = 36.2%

worst 10 classes:
  boy              0.480  (n=100)
  girl             0.530  (n=100)
  otter            0.540  (n=100)
  bowl             0.570  (n=100)
  seal             0.590  (n=100)
  possum           0.590  (n=100)
  bear             0.620  (n=100)
  mouse            0.640  (n=100)
  crocodile        0.640  (n=100)
  dolphin          0.640  (n=100)
best 5 classes:
  tractor          0.970
  motorcycle       0.960
  plain            0.940
  wardrobe         0.940
  orange           0.940

top confused pairs (true -> predicted):
  boy              -> man              18
  oak_tree         -> maple_tree       16
  girl             -> woman            16
  otter            -> seal             13
  maple_tree       -> oak_tree         13
  willow_tree      -> oak_tree         12
  worm             -> snake            11
  willow_tree      -> maple_tree       

## S6e - Error samples

Read the 12 highest-confidence misclassifications saved by S6d, undo the normalization,
and draw a grid. This figure is used in report section 6.3.


In [7]:
# S6e: error-sample grid (reads S6d's npz, no model rerun)
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from src.config import PROJECT_ROOT, RUNS_DIR

plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 150, "font.size": 8})
FIG = PROJECT_ROOT / "figs"
RUNS_FINAL = RUNS_DIR.parent / "runs_final"
assert (RUNS_FINAL / "error_samples.npz").exists(), "Run S6d first; it writes error_samples.npz"

d = np.load(RUNS_FINAL / "error_samples.npz", allow_pickle=False)
imgs, true, pred, conf = d["imgs"], d["true"], d["pred"], d["conf"]
names = [str(x) for x in d["fine_names"]]
MEAN = np.array([0.5071, 0.4866, 0.4409]).reshape(3, 1, 1)
STD = np.array([0.2673, 0.2564, 0.2762]).reshape(3, 1, 1)

fig, axes = plt.subplots(3, 4, figsize=(8.4, 6.6))
for ax, im, t, p, c in zip(axes.ravel(), imgs, true, pred, conf):
    img = np.clip(im * STD + MEAN, 0, 1).transpose(1, 2, 0)
    ax.imshow(img, interpolation="nearest")
    ax.set_title("true: %s\npred: %s (%.2f)" % (names[t], names[p], c), fontsize=7)
    ax.axis("off")
fig.suptitle("Highest-confidence misclassifications (final 3-seed ensemble, test10k)", fontsize=9)
fig.tight_layout()
fig.savefig(FIG / "fig_error_samples.png", bbox_inches="tight")
plt.close(fig)
print("wrote", FIG / "fig_error_samples.png")


wrote C:\Users\sigridwicks\Desktop\DL HW1\figs\fig_error_samples.png
